# 33 · Is the positional signal just misregistration?

Every result in notebooks 31–32 rests on the species-by-position statistic `T_spatial`. Its main
alternative explanation is **coordinate misregistration**. If human positions were shifted or
stretched relative to mouse, any gene with a positional gradient would show a spurious
species-by-position difference. That would also produce "human-high early, mouse-high late"
patterns and apparent direction reversals.

Notebook 31's relabeling control cannot detect this. A coordinate offset between species is
itself a species effect, and it cancels in mixed mouse+human groups. This notebook tests it
directly:

1. **Anatomical anchors.** Where do the reviewed S1→S2 and S2→S3 transitions fall on the
   coordinate in each specimen?
2. **Global registration.** Can any affine stretch or shift of the human coordinate make human and
   mouse gene curves agree much better?
3. **Anchored re-analysis.** Warp each human section so its segment transitions land on the mouse
   transitions, refit every gene, rerun the pathway screen, and compare with the original list.
4. **Gene-level check.** Can a per-gene shift explain each direction-reversing gene?

Cohort reminder: 2 control mice and 2 cortex sections from 1 male human donor. Descriptive.

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream31 = results_root / 'pt_pathway_method_selection'
output = results_root / 'pt_pathway_registration_sensitivity'
figures = output / 'figures'
figures.mkdir(parents=True, exist_ok=True)
for path in (upstream31 / 'final_pathway_table.csv', upstream31 / 'gene_positional_summary.csv'):
    if not path.is_file():
        raise FileNotFoundError(f'Run notebook 31 first: missing {path}')

NOTEBOOK_LOGIC_VERSION = '33.registration_sensitivity.1'
BASIS_DF, N_NULL, SEED, ALPHA = 6, 9999, 12, .05
print('Results folder:', output)

## 1 · Rebuild notebook 12's structures, genes and pathways

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.pathway_remodeling import (fit_nested_trajectories, gene_covariates, matching_strata,
                                            matched_pathway_tests)
from pseudospace.pathways import build_pathway_membership
from pseudospace.stage_cache import cached_payload, cached_frame, digest
import pseudospace.pathway_remodeling as remodeling

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
support = pt_obs.groupby('sample', observed=True).shared_pseudospace.quantile([.01, .99]).unstack()
coordinate = pt_obs.shared_pseudospace.to_numpy(float)
pt_obs = pt_obs[(coordinate >= support[.01].max()) & (coordinate <= support[.99].min())].copy()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
adata = rebuild_pt_expression(pt_obs, {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad'
                                       for v in expected.values() for n in v}, orthologs, target_sum=1e4)
position = adata.obs.shared_pseudospace.to_numpy(float)
human = adata.obs.comparison_species.astype(str).eq('human').to_numpy()
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).to_numpy()
names = expected['mouse'] + expected['human']
expression = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)
universe = gene_covariates(expression, position, human, specimen, adata.var_names)
universe['eligible'] = adata.var.measured_in_both_inputs.to_numpy() & universe.detection.ge(.02)
genes = universe.loc[universe.eligible, 'gene'].tolist()
gene_index = pd.Index(genes)
Y = expression[:, universe.eligible.to_numpy()]
covariates = universe.set_index('gene').loc[genes]
strata = matching_strata(covariates, bins=3)  # Original strata: only the gene statistic changes below.
coverage = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    universe.loc[adata.var.measured_in_both_inputs.to_numpy(), 'gene'], ortholog_map=orthologs,
    library_name=library, tested=genes, min_genes=1)
    for library in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')], ignore_index=True)
coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
coverage = coverage[coverage.n_tested.between(10, min(300, len(genes) - 1))]
gene_sets = {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}
final31 = pd.read_csv(upstream31 / 'final_pathway_table.csv')
gene_table31 = pd.read_csv(upstream31 / 'gene_positional_summary.csv', index_col=0)
if set(final31.pathway_id) != set(gene_sets) or list(gene_table31.index) != genes:
    raise ValueError('Inputs differ from notebook 31; rerun notebook 31 first.')
cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(remodeling.__file__)])
print(f'{len(position):,} structures · {len(genes):,} genes · {len(gene_sets):,} pathways')

## 2 · Where do the reviewed segment transitions fall in each specimen?

For adjacent segments, fit a logistic curve of "later segment" against position within each
specimen. The transition is where both labels are equally likely. The labels come from reviewed
marker-defined clusters, so they are an anatomical anchor that does not depend on how well
individual gene curves agree.

In [ ]:
from sklearn.linear_model import LogisticRegression

transition_rows = []
for name in names:
    for first, second in (('PT-S1', 'PT-S2'), ('PT-S2', 'PT-S3')):
        keep = (specimen == name) & np.isin(segment, [first, second])
        model = LogisticRegression(C=1e4).fit(position[keep][:, None], (segment[keep] == second).astype(int))
        transition_rows.append({'specimen': name, 'species': 'human' if name in expected['human'] else 'mouse',
                                'transition': f'{first[-2:]}→{second[-2:]}',
                                'position': float(-model.intercept_[0] / model.coef_[0, 0])})
transitions = pd.DataFrame(transition_rows)
transition_table = transitions.pivot(index='specimen', columns='transition', values='position')
transition_table.to_csv(output / 'segment_transitions_by_specimen.csv')
display(transition_table.round(3))
mouse_anchor = transitions[transitions.species.eq('mouse')].groupby('transition').position.mean()
print('Mouse anchors:', mouse_anchor.round(3).to_dict())

## 3 · Can a global stretch or shift make the species agree?

Fit the full model on the original coordinate and take each species' fitted curve on a fine grid.
For 500 genes with clear positional structure in **both** species, transform the human curve by
s → a + b·s and measure the mean Pearson correlation with the mouse curve over the overlap
(at least 80% of the coordinate). Correlation ignores level and scale, so it asks only whether
the shapes line up. A large gain would indicate misregistration; a small gain means the curves
genuinely differ in shape.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 9,
                     'axes.titlesize': 10, 'axes.titleweight': 'bold', 'pdf.fonttype': 42})
HUMAN, MOUSE, INK = '#D55E00', '#0072B2', '#333333'


def save(fig, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=200)


def common_grid(pos, n=121):
    lo = max(pos[specimen == name].min() for name in names)
    hi = min(pos[specimen == name].max() for name in names)
    return np.linspace(lo, hi, n)


fine = common_grid(position)
original_fit = cached_payload('original_fit_fine',
    lambda: dict(fit_nested_trajectories(Y, position, human, specimen, fine, basis_df=BASIS_DF)),
    root=cache, params={'basis_df': BASIS_DF, 'grid': fine},
    inputs={'Y': digest(Y), 'position': position, 'human': human, 'specimen': specimen}, code=code)
H, M = original_fit['human'], original_fit['mouse']
shared = np.minimum(H.std(axis=1), M.std(axis=1))
shared[(covariates.detection_mouse < .1).to_numpy() | (covariates.detection_human < .1).to_numpy()] = 0
registration_genes = np.argsort(shared)[::-1][:500]


def mean_shape_correlation(a, b, idx, min_overlap=.8):
    source = (fine - a) / b
    keep = (source >= fine[0]) & (source <= fine[-1])
    if keep.mean() < min_overlap:
        return np.nan
    h = np.array([np.interp(source[keep], fine, H[i]) for i in idx])
    m = M[idx][:, keep]
    h -= h.mean(axis=1, keepdims=True)
    m -= m.mean(axis=1, keepdims=True)
    return float(np.mean((h * m).sum(axis=1) / np.sqrt((h ** 2).sum(axis=1) * (m ** 2).sum(axis=1))))


shifts, stretches = np.linspace(-.12, .12, 25), np.linspace(.85, 1.15, 16)
surface = np.array([[mean_shape_correlation(a, b, registration_genes) for b in stretches] for a in shifts])
best = np.unravel_index(np.nanargmax(surface), surface.shape)
identity_r = mean_shape_correlation(0., 1., registration_genes)
registration_summary = pd.Series({'identity_mean_r': identity_r, 'best_mean_r': surface[best],
                                  'best_shift': shifts[best[0]], 'best_stretch': stretches[best[1]],
                                  'best_on_search_edge': best[0] in (0, len(shifts) - 1)
                                  or best[1] in (0, len(stretches) - 1)})
registration_summary.to_csv(output / 'global_registration_summary.csv')
display(registration_summary.to_frame('value'))

## 4 · Re-analysis on an anatomy-anchored human coordinate

Warp each human section piecewise-linearly so that its own S1→S2 and S2→S3 transitions move to
the mouse transitions. The ends of the coordinate stay fixed and mouse positions are unchanged.
Then refit all genes, recompute `T_spatial`, and rerun notebook 12's matched pathway test with
the same strata, 9,999 sets and seed. A pathway result that comes from misregistration at the
segment level should disappear here.

In [ ]:
def anchored_warp(values, own, target):
    """Monotone piecewise-linear map sending own S1→S2/S2→S3 transitions to the target ones."""
    knots_from = np.r_[0., own['S1→S2'], own['S2→S3'], 1.]
    knots_to = np.r_[0., target['S1→S2'], target['S2→S3'], 1.]
    if np.any(np.diff(knots_from) <= 0) or np.any(np.diff(knots_to) <= 0):
        raise ValueError('Segment transitions must be ordered inside (0, 1).')
    return np.interp(values, knots_from, knots_to)


warped = position.copy()
for name in expected['human']:
    own = transitions[transitions.specimen.eq(name)].set_index('transition').position
    mask = specimen == name
    warped[mask] = anchored_warp(position[mask], own, mouse_anchor)
check = []
for name in names:
    for first, second in (('PT-S1', 'PT-S2'), ('PT-S2', 'PT-S3')):
        keep = (specimen == name) & np.isin(segment, [first, second])
        model = LogisticRegression(C=1e4).fit(warped[keep][:, None], (segment[keep] == second).astype(int))
        check.append({'specimen': name, 'transition': f'{first[-2:]}→{second[-2:]}',
                      'position': float(-model.intercept_[0] / model.coef_[0, 0])})
display(pd.DataFrame(check).pivot(index='specimen', columns='transition', values='position').round(3)
        .rename_axis(columns='after warping'))

grid = common_grid(warped, 61)
warped_fit = cached_payload('warped_fit',
    lambda: dict(fit_nested_trajectories(Y, warped, human, specimen, grid, basis_df=BASIS_DF)),
    root=cache, params={'basis_df': BASIS_DF, 'grid': grid},
    inputs={'Y': digest(Y), 'position': warped, 'human': human, 'specimen': specimen}, code=code)
statistics = pd.DataFrame({'T_spatial_original': gene_table31.T_spatial,
                           'T_spatial_warped': warped_fit['T_spatial']}, index=genes)
warped_tests = cached_frame('warped_matched',
    lambda: matched_pathway_tests(statistics[['T_spatial_warped']].rename(columns={'T_spatial_warped': 'T_spatial'}),
                                  gene_sets, strata, n_null=N_NULL, seed=SEED),
    root=cache, params={'n_null': N_NULL, 'seed': SEED},
    inputs={'scores': statistics.T_spatial_warped, 'sets': gene_sets, 'strata': strata}, code=code)
warped_tests['warped_hit'] = warped_tests.effect.gt(0) & warped_tests.q_empirical.le(ALPHA)
comparison = final31[['pathway_id', 'name', 'theme', 'candidate', 'confident', 'pseudospace_class',
                      'effect_T_spatial', 'q_spatial']].merge(
    warped_tests[['pathway_id', 'effect', 'q_empirical', 'warped_hit']].rename(
        columns={'effect': 'effect_warped', 'q_empirical': 'q_warped'}), on='pathway_id', validate='one_to_one')
comparison.to_csv(output / 'pathway_original_vs_anchored.csv', index=False)
from scipy.stats import spearmanr
retained = pd.Series({
    'gene T_spatial Spearman (original vs anchored)': spearmanr(statistics.T_spatial_original,
                                                                statistics.T_spatial_warped).statistic,
    'pathway effect Spearman': spearmanr(comparison.effect_T_spatial, comparison.effect_warped).statistic,
    'spatial calls, original': int(comparison.candidate.sum()),
    'spatial calls, anchored': int(comparison.warped_hit.sum()),
    'original calls retained': int((comparison.candidate & comparison.warped_hit).sum()),
    'confident pathways retained': f'{int((comparison.confident & comparison.warped_hit).sum())} of '
                                   f'{int(comparison.confident.sum())}',
    'new-with-pseudospace retained': f'{int((comparison.pseudospace_class.eq("new with pseudospace") & comparison.warped_hit).sum())} of '
                                     f'{int(comparison.pseudospace_class.eq("new with pseudospace").sum())}'})
retained.rename('value').to_csv(output / 'anchored_reanalysis_summary.csv')
display(retained.to_frame('value'))
lost = comparison[comparison.confident & ~comparison.warped_hit]
display(lost[['name', 'theme', 'q_spatial', 'q_warped', 'effect_T_spatial', 'effect_warped']].round(3))

## 5 · Do the direction reversals and the early/late split survive?

Apply notebook 31's reversal rule to the anchored fit. Separately, give each original reversing
gene its own best shift of the human curve (−0.15 to +0.15 coordinate units) and ask whether the
shifted difference still has clear human-high and mouse-high stretches. A per-gene shift is very
flexible, so a reversal that survives it is not a registration artefact.

In [ ]:
def reverses(z_row, d_row, threshold=3., effect=.25, run=3):
    def longest(mask):
        best = current = 0
        for v in mask:
            current = current + 1 if v else 0
            best = max(best, current)
        return best
    up = (z_row >= threshold) & (d_row >= effect)
    down = (z_row <= -threshold) & (d_row <= -effect)
    return longest(up) >= run and longest(down) >= run


warped_reversing = pd.Series([reverses(z, d) for z, d in zip(warped_fit['z'], warped_fit['delta'])], index=genes)
original_reversing = gene_table31.reverses_along_PT.astype(bool)
original_fit61 = fit_nested_trajectories(Y, position, human, specimen, common_grid(position, 61), basis_df=BASIS_DF)


def survives_best_shift(i, effect=.25, run=3):
    """Does a per-gene shift leave clear opposite-signed stretches in the difference curve?"""
    g = common_grid(position, 61)
    h, m, se = original_fit61['human'][i], original_fit61['mouse'][i], original_fit61['se'][i]
    best = None
    for shift in np.linspace(-.15, .15, 31):
        source = g - shift
        keep = (source >= g[0]) & (source <= g[-1])
        if keep.mean() < .7:
            continue
        d = np.interp(source[keep], g, h) - m[keep]
        loss = np.mean((d - d.mean()) ** 2)
        if best is None or loss < best[0]:
            best = (loss, shift, d, se[keep])
    _, shift, d, s = best
    return shift, reverses(d / s, d, effect=effect, run=run)


rows = []
for gene in original_reversing[original_reversing].index:
    shift, still = survives_best_shift(gene_index.get_loc(gene))
    rows.append({'gene': gene, 'best_shift': shift, 'reverses_after_best_shift': still,
                 'reverses_on_anchored_coordinate': bool(warped_reversing[gene])})
reversal_check = pd.DataFrame(rows)
reversal_check.to_csv(output / 'reversing_gene_registration_check.csv', index=False)
confident_ids = comparison.loc[comparison.confident, 'pathway_id']
peaks = []
for label, fit_ in (('original', original_fit61), ('anchored', warped_fit)):
    g = common_grid(position if label == 'original' else warped, 61)
    local = remodeling.local_pathway_curves(fit_['z'], genes, {p: gene_sets[p] for p in confident_ids}, g)
    at_peak = local.loc[local.groupby('pathway_id').divergence.idxmax()]
    peaks.extend({'coordinate': label, 'pathway_id': r.pathway_id, 'peak': float(r.position),
                  'human_high_at_peak': bool(r.median_member_z > 0)} for r in at_peak.itertuples())
peaks = pd.DataFrame(peaks)
split = peaks.groupby(['coordinate', 'human_high_at_peak']).peak.agg(['size', 'median']).round(3)
split.to_csv(output / 'early_late_split.csv')
summary5 = pd.Series({'reversing genes, original': int(original_reversing.sum()),
                      'reversing genes, anchored coordinate': int(warped_reversing.sum()),
                      'original reversals kept on anchored coordinate': int(reversal_check.reverses_on_anchored_coordinate.sum()),
                      'original reversals surviving a per-gene best shift': int(reversal_check.reverses_after_best_shift.sum())})
display(summary5.to_frame('genes'))
display(split)

## 6 · Figure 8 · Registration sensitivity at a glance

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), layout='constrained',
                         gridspec_kw={'width_ratios': [1, 1, 1, 1]})
ax = axes[0]
for name in names:
    color = HUMAN if name in expected['human'] else MOUSE
    marker = 'o' if name in (expected['human'][0], expected['mouse'][0]) else 's'
    row = transition_table.loc[name]
    ax.scatter(row.values, [0, 1], color=color, marker=marker, s=40, label=name)
ax.set_yticks([0, 1], transition_table.columns)
ax.set(xlabel='Position on shared coordinate', ylim=(-.6, 1.6))
ax.legend(frameon=False, fontsize=7, loc='center right')
ax.set_title('Segment transitions', loc='left')
ax = axes[1]
image = ax.imshow(surface, aspect='auto', origin='lower', cmap='Greys',
                  extent=(stretches[0], stretches[-1], shifts[0], shifts[-1]))
ax.scatter([1], [0], marker='+', s=80, color=HUMAN, label=f'identity r = {identity_r:.3f}')
ax.scatter([stretches[best[1]]], [shifts[best[0]]], marker='x', s=50, color=MOUSE,
           label=f'best r = {surface[best]:.3f}')
ax.set(xlabel='Stretch b', ylabel='Shift a')
ax.legend(frameon=False, fontsize=7, loc='upper right')
fig.colorbar(image, ax=ax, label='Mean shape correlation')
ax.set_title('Global registration gain', loc='left')
ax = axes[2]
ax.scatter(comparison.effect_T_spatial, comparison.effect_warped, s=8, color='#BBBBBB', rasterized=True)
conf = comparison[comparison.confident]
kept = conf.warped_hit
ax.scatter(conf.effect_T_spatial[kept], conf.effect_warped[kept], s=14, color=HUMAN, label='confident, kept')
ax.scatter(conf.effect_T_spatial[~kept], conf.effect_warped[~kept], s=18, color=INK, marker='x', label='confident, lost')
lim = [min(comparison.effect_T_spatial.min(), comparison.effect_warped.min()),
       max(comparison.effect_T_spatial.max(), comparison.effect_warped.max())]
ax.plot(lim, lim, color='#999999', lw=.8)
ax.set(xlabel='Spatial effect, original', ylabel='Spatial effect, anchored')
ax.legend(frameon=False, fontsize=7, loc='upper left')
ax.set_title('Pathways after re-anchoring', loc='left')
ax = axes[3]
for i, label in enumerate(('original', 'anchored')):
    for high, color in ((True, HUMAN), (False, MOUSE)):
        values = peaks[(peaks.coordinate == label) & (peaks.human_high_at_peak == high)].peak
        jitter = np.random.default_rng(0).uniform(-.12, .12, len(values))
        ax.scatter(np.full(len(values), i) + (-.18 if high else .18) + jitter * .5, values, s=8,
                   color=color, alpha=.7)
ax.set_xticks([0, 1], ['original', 'anchored'])
ax.set_ylabel('Position of largest divergence')
ax.set_title('Early/late split', loc='left')
ax.scatter([], [], color=HUMAN, s=10, label='human-high at peak')
ax.scatter([], [], color=MOUSE, s=10, label='mouse-high at peak')
ax.legend(frameon=False, fontsize=7, loc='upper center', bbox_to_anchor=(.5, -.18), ncol=2)
save(fig, 'fig8_registration_sensitivity')
plt.show()
plt.close(fig)
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'mouse_anchor': mouse_anchor.to_dict(),
            'transitions': transition_table.round(4).to_dict(orient='index'),
            'global_registration': registration_summary.to_dict(),
            'anchored_reanalysis': retained.astype(str).to_dict(), 'reversals': summary5.to_dict()}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))

## 7 · Coordinate-free confirmation from segment DESeq2

The reviewed S1/S2/S3 labels do not depend on the coordinate. A reversal is confirmed without
any coordinate if segment-level DESeq2 (notebook 31, specimen pseudobulk) finds the species
difference significantly positive in one segment and significantly negative in another
(padj ≤ 0.05 both ways). We also mark genes whose **whole-PT** comparison is not significant:
there, the conventional answer is "no species difference" while two segments differ in opposite
directions.

In [ ]:
de = pd.read_csv(upstream31 / 'conventional_deseq2_all_partitions.csv')
de = de[de.partition.eq('species')]
lfc = de.pivot(index='gene', columns='scope', values='log2FoldChange')
padj = de.pivot(index='gene', columns='scope', values='padj')
segments = ['PT-S1', 'PT-S2', 'PT-S3']


def segment_opposite(gene):
    if gene not in lfc.index:
        return np.nan
    pairs = [(lfc.loc[gene, s], padj.loc[gene, s]) for s in segments]
    return any(v > 0 and p <= ALPHA for v, p in pairs) and any(v < 0 and p <= ALPHA for v, p in pairs)


reversal_check['segment_DE_opposite_signs'] = [segment_opposite(g) for g in reversal_check.gene]
reversal_check['whole_PT_padj'] = [padj.loc[g, 'whole_PT'] if g in padj.index else np.nan for g in reversal_check.gene]
reversal_check['whole_PT_not_significant'] = reversal_check.whole_PT_padj.gt(ALPHA)
reversal_check.to_csv(output / 'reversing_gene_registration_check.csv', index=False)
confirmed = reversal_check.segment_DE_opposite_signs.fillna(False).astype(bool)
summary7 = pd.Series({'reversing genes': len(reversal_check),
                      'confirmed by segment DESeq2 (opposite significant signs)': int(confirmed.sum()),
                      '... and robust to a per-gene shift': int((confirmed & reversal_check.reverses_after_best_shift).sum()),
                      '... whose whole-PT comparison is not significant': int((confirmed & reversal_check.whole_PT_not_significant).sum())})
summary7.rename('genes').to_csv(output / 'reversal_confirmation_summary.csv')
display(summary7.to_frame('genes'))
hidden = reversal_check[confirmed & reversal_check.whole_PT_not_significant].gene.tolist()
print('Whole-PT says "no difference"; segments differ in opposite directions:', ', '.join(hidden))

### Figure 9 · Opposite regional differences that whole-PT analysis averages away

Fitted human and mouse curves on the original coordinate, with each specimen's binned means as
dots, beside the whole-PT and per-segment DESeq2 log2 fold changes (* padj ≤ 0.05).

In [ ]:
example_genes = [g for g in ('Slc22a6', 'Slc13a3', 'Cyp24a1', 'Slc22a7') if g in set(reversal_check.gene)]
g61 = common_grid(position, 61)
edges = np.linspace(g61[0], g61[-1], 13)
centres = (edges[:-1] + edges[1:]) / 2
binned = np.clip(np.searchsorted(edges, position, side='right') - 1, 0, len(centres) - 1)
spans = {s: np.quantile(position[segment == s], [.1, .9]) for s in segments}
fig, axes = plt.subplots(2, len(example_genes), figsize=(3.2 * len(example_genes), 5.2), layout='constrained',
                         gridspec_kw={'height_ratios': [1.6, 1]})
for k, gene in enumerate(example_genes):
    i = gene_index.get_loc(gene)
    ax = axes[0, k]
    for s, (a, b) in spans.items():
        ax.axvspan(a, b, color='#000000', alpha=.04 + .03 * segments.index(s), lw=0)
        ax.text((a + b) / 2, 1.01, s[-2:], transform=ax.get_xaxis_transform(), ha='center', va='bottom',
                fontsize=7.5, color='#555555')
    ax.plot(g61, original_fit61['human'][i], color=HUMAN, lw=2, label='human')
    ax.plot(g61, original_fit61['mouse'][i], color=MOUSE, lw=2, label='mouse')
    column = Y[:, i].toarray().ravel()
    for name in names:
        means = [column[(specimen == name) & (binned == b)].mean() if np.any((specimen == name) & (binned == b))
                 else np.nan for b in range(len(centres))]
        ax.scatter(centres, means, s=9, color=HUMAN if name in expected['human'] else MOUSE, alpha=.75, zorder=3)
    ax.set_title(gene, loc='left', pad=14)
    ax.set(xlabel='PT pseudospace', xlim=(g61[0], g61[-1]))
    if k == 0:
        ax.set_ylabel('Log-normalized expression')
        ax.legend(frameon=False, fontsize=7.5, loc='upper left')
    ax = axes[1, k]
    values = lfc.loc[gene, ['whole_PT', *segments]]
    ax.bar(range(4), values.to_numpy(), color=['#555555'] + [HUMAN if v > 0 else MOUSE for v in values[1:]],
           width=.7)
    for j, p in enumerate(padj.loc[gene, ['whole_PT', *segments]]):
        label = '*' if p <= ALPHA else 'ns'
        ax.text(j, values.iloc[j], label, ha='center', va='bottom' if values.iloc[j] >= 0 else 'top', fontsize=8)
    ax.axhline(0, color='#999999', lw=.7)
    ax.set_xticks(range(4), ['whole PT', 'S1', 'S2', 'S3'])
    if k == 0:
        ax.set_ylabel('DESeq2 log2FC\n(human − mouse)')
save(fig, 'fig9_opposite_regional_differences')
plt.show()
plt.close(fig)
manifest['reversal_confirmation'] = summary7.to_dict()
manifest['whole_PT_hidden_reversals'] = hidden
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))

## 8 · Are late-PT results driven by mouse outer-stripe S3?

Both human sections are cortex. The mouse sections are whole-kidney sections, so mouse S3
includes straight segments in the outer stripe of the outer medulla, which the human sections
cannot contain. A late-PT species difference could therefore reflect sampled region rather than
species.

Glomeruli lie only in cortex, so distance to glomeruli is a physical cortex proxy. Following
notebook 16, a structure's depth is its mean distance to the three nearest reviewed glomeruli
of the same specimen. Centroids are verified against the current fine GeoJSON first (the
mandatory mouse centroid guard). We call a mouse S3 structure **cortical-like** when it lies no
deeper than the 95th percentile of the same specimen's S1 structures (S1 is cortical by
definition). The rest of mouse S3 is excluded, the models are refit, and the pathway screen is
rerun.

In [ ]:
from scipy.spatial import cKDTree
from pseudospace.repeated_inputs import _read_and_validate_segmentation

pass1 = ad.read_h5ad(results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad', backed='r')
anchors_all = pass1.obs[['sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid']].copy()
pass1.file.close()
depth = np.full(len(position), np.nan)
depth_rows = []
for name in expected['mouse']:
    native = ad.read_h5ad(data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad', backed='r')
    source_obs = native.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
    native.file.close()
    mask = specimen == name
    ids = adata.obs_names[mask].str[len(name) + 1:]
    structures = source_obs.loc[ids]
    _read_and_validate_segmentation(data_root, name, structures)
    glomeruli = anchors_all[anchors_all['sample'].astype(str).eq(name) & anchors_all.coarse_class.astype(str).eq('Glomerulus')]
    _read_and_validate_segmentation(data_root, name, glomeruli)
    distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
        structures[['x_centroid', 'y_centroid']].to_numpy(float), k=3)
    depth[mask] = distance.mean(axis=1)
    s1_limit = np.quantile(depth[mask & (segment == 'PT-S1')], .95)
    for label in segments:
        values = depth[mask & (segment == label)]
        depth_rows.append({'specimen': name, 'segment': label, 'n': len(values),
                           'median_depth_px': float(np.median(values)), 'S1_p95_limit_px': float(s1_limit),
                           'fraction_cortical_like': float(np.mean(values <= s1_limit))})
depth_table = pd.DataFrame(depth_rows)
depth_table.to_csv(output / 'mouse_depth_by_segment.csv', index=False)
display(depth_table.round(3))
limits = depth_table.groupby('specimen').S1_p95_limit_px.first()
far_mouse_s3 = np.zeros(len(position), bool)
for name in expected['mouse']:
    far_mouse_s3 |= (specimen == name) & (segment == 'PT-S3') & (depth > limits[name])
keep = ~far_mouse_s3
print(f'Excluded {far_mouse_s3.sum():,} deep mouse S3 structures; {keep.sum():,} remain.')

kept_grid = np.linspace(max(position[keep & (specimen == n)].min() for n in names),
                        min(position[keep & (specimen == n)].max() for n in names), 61)
cortical_fit = cached_payload('cortical_fit',
    lambda: dict(fit_nested_trajectories(Y[keep], position[keep], human[keep], specimen[keep], kept_grid,
                                         basis_df=BASIS_DF)),
    root=cache, params={'basis_df': BASIS_DF, 'grid': kept_grid},
    inputs={'Y': digest(Y), 'keep': keep, 'position': position}, code=code)
cortical_tests = cached_frame('cortical_matched',
    lambda: matched_pathway_tests(pd.DataFrame({'T_spatial': cortical_fit['T_spatial']}, index=genes),
                                  gene_sets, strata, n_null=N_NULL, seed=SEED),
    root=cache, params={'n_null': N_NULL, 'seed': SEED},
    inputs={'scores': cortical_fit['T_spatial'], 'sets': gene_sets, 'strata': strata}, code=code)
cortical_tests['cortical_hit'] = cortical_tests.effect.gt(0) & cortical_tests.q_empirical.le(ALPHA)
comparison = comparison.merge(cortical_tests[['pathway_id', 'effect', 'q_empirical', 'cortical_hit']].rename(
    columns={'effect': 'effect_cortical', 'q_empirical': 'q_cortical'}), on='pathway_id', validate='one_to_one')
comparison.to_csv(output / 'pathway_original_vs_anchored.csv', index=False)
is_new = comparison.pseudospace_class.eq('new with pseudospace')
summary8 = pd.Series({
    'gene T_spatial Spearman (original vs cortical-like)': spearmanr(statistics.T_spatial_original,
                                                                     cortical_fit['T_spatial']).statistic,
    'spatial calls, cortical-like mouse S3': int(comparison.cortical_hit.sum()),
    'confident pathways retained': f'{int((comparison.confident & comparison.cortical_hit).sum())} of {int(comparison.confident.sum())}',
    'new-with-pseudospace retained': f'{int((is_new & comparison.cortical_hit).sum())} of {int(is_new.sum())}',
    'confident retained under both checks': f'{int((comparison.confident & comparison.cortical_hit & comparison.warped_hit).sum())} of {int(comparison.confident.sum())}'})
summary8.rename('value').to_csv(output / 'cortical_restriction_summary.csv')
display(summary8.to_frame('value'))
display(comparison[comparison.confident & ~comparison.cortical_hit][['name', 'theme', 'pseudospace_class', 'q_spatial', 'q_cortical']].round(3))
manifest['cortical_restriction'] = summary8.astype(str).to_dict()
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))

## 9 · The robust list for the paper

A confident pathway (notebook 31) is **robust** when it is still called after anatomical
re-anchoring of the human coordinate **and** after restricting mouse S3 to cortical-like
structures. Pathways lost only under the cortical restriction are flagged as region-sensitive:
their late-PT signal may depend on mouse outer-stripe S3, or on the power lost by removing most
of mouse S3. They are not refuted.

In [ ]:
final_robust = final31.merge(comparison[['pathway_id', 'warped_hit', 'q_warped', 'cortical_hit', 'q_cortical']],
                             on='pathway_id', validate='one_to_one')
final_robust = final_robust[final_robust.confident.astype(bool)].copy()
final_robust['robustness'] = np.select(
    [final_robust.warped_hit & final_robust.cortical_hit, final_robust.warped_hit & ~final_robust.cortical_hit,
     ~final_robust.warped_hit & final_robust.cortical_hit],
    ['robust', 'region-sensitive', 'registration-sensitive'], 'sensitive to both')
final_robust.sort_values(['robustness', 'q_spatial']).to_csv(output / 'confident_pathways_with_robustness.csv', index=False)
display(pd.crosstab(final_robust.robustness, final_robust.pseudospace_class))